In [ ]:
# !pip install optuna

In [ ]:
# ! pip install kagglehub[pandas-datasets]

In [ ]:
# ! pip install torchinfo

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot  as plt
import seaborn as sns
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import optuna
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from imblearn.over_sampling import SMOTE

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)


In [5]:
# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [4]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
# For this dataset, the main file is 'ai4i2020.csv'
file_path = "ai4i2020.csv"

# Load the latest version using the recommended dataset_load method
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "stephanmatzka/predictive-maintenance-dataset-ai4i-2020",
  file_path
)

100%|██████████| 510k/510k [00:01<00:00, 448kB/s]


In [ ]:
# # Save to your Drive
# df.to_csv('/content/drive/MyDrive/Colab Notebooks/Predictive Maintenance Dataset (AI4I 2020)/ai4i2020_saved.csv', index=False)
# print("✅ Saved to Google Drive!")

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
df=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Predictive Maintenance Dataset (AI4I 2020)/ai4i2020_saved.csv')
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [9]:
# Drop non-predictive columns
df = df.drop(columns=['UDI', 'Product ID'])

In [10]:
pd.DataFrame({"Missing value":df.isna().sum(),"Datatype":df.dtypes})

,Missing value,Datatype
Type,0,object
Air temperature [K],0,float64
Process temperature [K],0,float64
Rotational speed [rpm],0,int64
Torque [Nm],0,float64
Tool wear [min],0,int64
Machine failure,0,int64
TWF,0,int64
HDF,0,int64
PWF,0,int64


In [11]:
X = df.drop('Machine failure', axis=1)
X.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],TWF,HDF,PWF,OSF,RNF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0


In [12]:
y=df[["Machine failure"]]
y.head()

,Machine failure
0,0
1,0
2,0
3,0
4,0


In [13]:
df["Machine failure"].value_counts()

,count
Machine failure,
0,9661
1,339


### This is highly imbalanced data for this we should balance the data using SMOTE

In [15]:
# Stratified split to preserve failure ratio
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [16]:
# Encode categorical feature
le = LabelEncoder()
X_train['Type'] = le.fit_transform(X_train['Type'])
X_test['Type'] = le.transform(X_test['Type'])

In [17]:
# Scale features
scaler = StandardScaler()
X_train_resampled = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [18]:
# Apply SMOTE to training data only
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)


In [20]:
y_train_resampled.value_counts()

,count
Machine failure,
0,7729
1,7729


In [57]:
# Custom Dataset and data loader
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = torch.tensor(features.values if hasattr(features, 'values') else features, dtype=torch.float32)
        self.labels = torch.tensor(labels.values.ravel() if hasattr(labels, 'values') else labels.ravel(), dtype=torch.long)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, index):
        return self.features[index],self.labels[index]

In [58]:
train_dataset = CustomDataset(X_train_resampled, y_train_resampled)

In [59]:
test_dataset = CustomDataset(X_test, y_test)

In [60]:
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [61]:
len(train_loader)

484

In [62]:
class PredictiveMaintenanceANN(nn.Module):
    def __init__(self,num_features):
        super().__init__()
        self.model=nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self,x):
        return self.model(x)

In [63]:
# set learning rate and epochs
epochs = 100
learning_rate = 0.1

In [64]:
# instatiate the model
model=PredictiveMaintenanceANN(X_train_resampled.shape[1])
model

PredictiveMaintenanceANN(
  (model): Sequential(
    (0): Linear(in_features=11, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [65]:
from torchinfo import summary

summary(model, input_size=(1, 11))

Layer (type:depth-idx)                   Output Shape              Param #
PredictiveMaintenanceANN                 [1, 1]                    --
├─Sequential: 1-1                        [1, 1]                    --
│    └─Linear: 2-1                       [1, 128]                  1,536
│    └─ReLU: 2-2                         [1, 128]                  --
│    └─Linear: 2-3                       [1, 64]                   8,256
│    └─ReLU: 2-4                         [1, 64]                   --
│    └─Linear: 2-5                       [1, 1]                    65
Total params: 9,857
Trainable params: 9,857
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.01
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.04
Estimated Total Size (MB): 0.04

In [66]:
# Loss Function
criterion = nn.BCEWithLogitsLoss()

# optimizer
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)


In [67]:
# training loop

for epoch in range(epochs):

    total_epoch_loss = 0

    for batch_features,batch_labels in train_loader:
        # forward pass
        outputs = model(batch_features)

        # calculate loss
        loss = criterion(outputs, batch_labels)

        # back pass
        optimizer.zero_grad()
        loss.backward()

        # update grads
        optimizer.step()

        total_epoch_loss = total_epoch_loss + loss.item()
    avg_loss = total_epoch_loss/len(train_loader)
    print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')

ValueError: Target size (torch.Size([32])) must be the same as input size (torch.Size([32, 1]))